# STEP 10: OpenDART 근거로 LLM 자연어 답변 만들기

이 노트북은 OpenDART가 가져온 재무 수치와 LLM의 자연어 답변 역할을 분리해 확인합니다.

- OpenDART와 Python: 사실 데이터 조회
- LLM: 제공된 사실 데이터를 읽고 사람이 읽기 쉬운 답변 작성

이번에는 Router, Tool Calling, 다중 기업 비교, RAG, Agent를 새로 추가하지 않습니다. Router의 역할은 09번에서 확인했으므로, 기업명과 사업연도는 사람이 직접 입력합니다.

## 1. 이번 workflow

```text
입력한 기업명·사업연도
  → fetch_financials: 실제 OpenDART 조회
  → generate_answer: 조회 결과만 근거로 LLM 답변 생성
  → answer 출력
```

여기서 LLM은 API를 호출하지 않습니다. `financial_data`를 Prompt에 넣어 주어야만 그 수치를 바탕으로 답변할 수 있습니다.

## 2. 환경 변수와 프로젝트 경로

`.env`에서 OpenDART API 키를 읽습니다. 키 값은 출력하지 않습니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
api_key = os.getenv("OPENDART_API_KEY")

if not api_key or api_key == "your_opendart_api_key":
    raise ValueError(".env의 OPENDART_API_KEY를 실제 API 키로 바꿔 주세요.")

print("프로젝트 경로와 OpenDART API 키를 준비했습니다.")

## 3. 필요한 구성요소 불러오기

이번 LLM 호출은 Structured Output이 아닌 일반 `invoke()`입니다. 따라서 반환값은 Pydantic 객체가 아니라 LLM이 작성한 메시지 내용입니다.

In [ ]:
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

from src.dart.client import get_financial_summary

## 4. 사람이 입력할 조회 조건

09번과 달리 LLM Router를 사용하지 않습니다. 마지막 답변 생성 역할만 분리해서 보기 위해 조회 조건을 직접 넣습니다.

In [ ]:
company_name = "삼성전자"
business_year = 2025
question = "매출액, 영업이익, 당기순이익을 간단히 설명해줘."

print(f"기업명: {company_name}")
print(f"사업연도: {business_year}")
print(f"질문: {question}")

## 5. State 정의

`financial_data`는 OpenDART 조회 Node가 추가하고, `answer`는 LLM 답변 Node가 추가합니다.

In [ ]:
class AnswerWorkflowState(TypedDict, total=False):
    company_name: str
    business_year: int
    question: str
    financial_data: dict[str, str | int | None]
    answer: str

## 6. OpenDART 조회 Node

이 Node가 사실 데이터의 출처입니다. LLM보다 먼저 실행됩니다.

In [ ]:
def fetch_financials(
    state: AnswerWorkflowState,
) -> dict[str, dict[str, str | int | None]]:
    financial_data = get_financial_summary(
        corp_name=state["company_name"],
        business_year=state["business_year"],
        api_key=api_key,
    )
    return {"financial_data": financial_data}

## 7. LLM 답변 생성 Node

System Prompt는 LLM의 역할과 제한을 정합니다. Human Prompt에는 사용자 질문과 OpenDART 근거 데이터를 함께 넣습니다.

중요: 이 Prompt는 근거 밖의 수치·원인을 만들지 말라고 지시할 뿐입니다. LLM의 사실성은 아직 평가하거나 검증하지 않습니다.

In [ ]:
llm = ChatOllama(
    model="qwen3.6:27b",
    base_url="http://localhost:11434",
    temperature=0,
)

def generate_answer(state: AnswerWorkflowState) -> dict[str, str]:
    data = state["financial_data"]
    evidence = "\n".join(
        f"- {label}: {value if value is not None else '조회되지 않음'}"
        for label, value in data.items()
    )

    messages = [
        SystemMessage(
            content=(
                "당신은 기업 재무정보를 설명하는 도우미입니다. "
                "제공된 OpenDART 근거 데이터만 사용하세요. "
                "근거에 없는 수치, 원인, 비교, 단위 추정이나 단위 변환을 하지 마세요."
            )
        ),
        HumanMessage(
            content=(
                f"사용자 질문: {state['question']}\n\n"
                f"OpenDART 근거 데이터:\n{evidence}\n\n"
                "위 근거를 바탕으로 한국어로 간단히 답변해줘."
            )
        ),
    ]
    response = llm.invoke(messages)
    return {"answer": str(response.content)}

## 8. Graph 연결

이번 순서는 고정입니다. 반드시 OpenDART 조회가 끝난 뒤 LLM이 답변을 작성하도록 연결합니다.

In [ ]:
workflow = StateGraph(AnswerWorkflowState)

workflow.add_node("fetch_financials", fetch_financials)
workflow.add_node("generate_answer", generate_answer)

workflow.add_edge(START, "fetch_financials")
workflow.add_edge("fetch_financials", "generate_answer")
workflow.add_edge("generate_answer", END)

app = workflow.compile()

## 9. Graph 구조 확인

실행 전에 조회 Node가 답변 Node보다 앞에 있는지 확인합니다.

In [ ]:
print(app.get_graph().draw_mermaid())

## 10. Workflow 실행

이 셀에서 OpenDART 조회와 LLM 자연어 답변 생성이 차례로 실행됩니다.

In [ ]:
result = app.invoke(
    {
        "company_name": company_name,
        "business_year": business_year,
        "question": question,
    }
)

## 11. OpenDART 근거와 LLM 답변을 나란히 확인

답변을 읽기 전에 원본 근거 데이터를 먼저 확인하세요. 현재 단계에서는 사람이 근거와 답변이 맞는지 비교합니다.

In [ ]:
print("OpenDART 근거 데이터:")
for label, value in result["financial_data"].items():
    print(f"- {label}: {value}")

print("\nLLM 자연어 답변:")
print(result["answer"])


## 12. 최소 확인

이 검사는 API 조회 결과와 LLM 답변 문자열이 State에 들어왔는지만 확인합니다. 답변의 정확도·충실도 평가는 이후 별도 평가 단계에서 다룹니다.

In [ ]:
assert result["financial_data"]["기업명"] == "삼성전자"
assert result["financial_data"]["사업연도"] == 2025
assert isinstance(result["answer"], str) and result["answer"].strip()

print("OpenDART 근거 데이터와 LLM 자연어 답변이 State에 연결되었습니다.")

## 이번 단계에서 이해할 점

- 09번의 LLM은 질문을 작업 지시로 바꾸는 Router였습니다.
- 이번 LLM은 이미 조회된 근거 데이터를 자연어로 설명하는 Answer Generator입니다.
- Structured Output은 LLM 결과를 정해진 필드로 받을 때 사용했고, 이번 일반 `invoke()`는 자연어 답변을 받을 때 사용했습니다.
- 자연어 답변을 만들었다고 사실성이 자동 보장되지는 않습니다. OpenDART 근거와 답변의 일치 여부는 이후 평가할 대상입니다.